## 介紹 

本課程將涵蓋： 
- 什麼是函數呼叫以及其使用情境 
- 如何使用 Azure OpenAI 建立函數呼叫 
- 如何將函數呼叫整合到應用程式中 

## 學習目標 

完成本課程後，你將會知道如何並理解： 

- 使用函數呼叫的目的 
- 使用 Azure OpenAI 服務設定函數呼叫 
- 為你的應用程式使用案例設計有效的函數呼叫 


## 理解函式呼叫

在本課程中，我們想為我們的教育新創公司建立一個功能，讓使用者可以使用聊天機器人來尋找技術課程。我們會根據他們的技能水平、目前職位和感興趣的技術來推薦課程。

為了完成這個目標，我們將結合使用：
 - `Azure Open AI` 來為使用者創建聊天體驗
 - `Microsoft Learn Catalog API` 協助使用者根據需求尋找課程
 - `Function Calling` 將使用者的查詢傳送到函式以發出 API 請求。

開始之前，讓我們先看看為什麼我們會想使用函式呼叫：


### 為什麼要使用 Function Calling 

如果你已經完成了本課程中的其他課程，你可能已經了解使用大型語言模型（LLM）的強大功能。希望你也能看到它們的一些限制。 

Function Calling 是 Azure Open AI 服務的一項功能，用於克服以下限制： 
1) 一致的回應格式 
2) 能在聊天環境中使用應用程式其他來源的數據 

在 Function Calling 出現之前，LLM 的回應是無結構且不一致的。開發人員需要撰寫複雜的驗證程式碼，以確保能夠處理每種回應變體。 

用戶無法得到像「斯德哥爾摩現在的天氣如何？」這樣的答案。這是因為模型的資料僅限於訓練時用的時間點。 

我們來看下面的例子，說明這個問題： 

假設我們想建立一個學生資料庫，以便推薦適合的課程給他們。以下我們有兩個學生的描述，它們所包含的資料非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此發送給大型語言模型（LLM）以解析數據。這可以在之後用於我們的應用程式中，發送到 API 或存儲在數據庫中。 

讓我們創建兩個相同的提示，以指示 LLM 我們感興趣的信息： 


我們想將此發送給大型語言模型（LLM），以解析對我們產品重要的部分。因此，我們可以創建兩個相同的提示來指示大型語言模型： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


建立這兩個提示後，我們會使用 `client.responses.create` 將它們發送到 LLM。我們將提示儲存在 `input` 變數中，並將角色指定為 `user`。這是為了模擬使用者向聊天機器人輸入訊息。 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

現在我們可以同時向 LLM 發送兩個請求，並檢查我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



雖然提示詞相同而且描述也相似，我們仍然可能獲得不同格式的 `Grades` 屬性。 

如果你多次執行上述單元格，格式可能是 `3.7` 或 `3.7 GPA`。 

這是因為大型語言模型（LLM）接受以書面提示形式存在的非結構化數據，並且也返回非結構化數據。我們需要有結構化格式，才能確定在存儲或使用這些數據時會得到什麼。 

透過使用功能調用，我們可以確保收到結構化數據。使用功能調用時，LLM 並不會實際調用或執行任何函數。相反，我們為 LLM 創建一個結構，讓它依照此結構回應。然後我們使用那些結構化回應來決定在我們的應用中要執行哪個函數。  
 


![函數調用流程圖](../../../../translated_images/zh-HK/Function-Flow.083875364af4f4bb.webp)


我們然後可以將函數返回的結果傳回給 LLM。LLM 隨後會使用自然語言回應，以回答用戶的查詢。


### 使用函數調用的案例

<strong>呼叫外部工具</strong>
聊天機器人在回答用戶問題方面表現優秀。透過使用函數調用，聊天機器人可以利用用戶的訊息來完成特定任務。例如，學生可以請聊天機器人「發送郵件給我的導師，說我需要更多這科目的協助」。這可以呼叫 `send_email(to: string, body: string)` 函數


**建立 API 或資料庫查詢**
使用者可以使用自然語言尋找資訊，並將其轉換為格式化的查詢或 API 請求。例如老師詢問「哪些學生完成了上一個作業」，可以呼叫名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數


<strong>建立結構化資料</strong>
使用者可以將一段文字或 CSV 利用大型語言模型提取重要資訊。例如，學生將維基百科關於和平協議的文章轉換為 AI 記憶卡。這可以透過使用名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函數完成


## 2. 建立你的第一個函數呼叫 

建立函數呼叫的過程包括三個主要步驟： 
1. 使用你的函數列表和用戶訊息呼叫 Chat Completions API 
2. 讀取模型的回應以執行動作，例如執行函數或 API 呼叫 
3. 使用函數的回應再次呼叫 Chat Completions API，利用該資訊為用戶產生回應。 


![函數調用流程](../../../../translated_images/zh-HK/LLM-Flow.3285ed8caf4796d7.webp)


### 函數呼叫的元素

#### 使用者輸入

第一步是建立一則使用者訊息。這可以透過取得文字輸入的數值動態指派，或是在這裡直接指派一個數值。如果這是您第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終使用者）。對於函數呼叫，我們會將其指定為 `user` 並給出一個範例問題。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。

接下來我們將定義一個函式及該函式的參數。我們這裡只會使用一個名為 `search_courses` 的函式，但你可以建立多個函式。

<strong>重要</strong>：函式會被包含在傳送到 LLM 的系統訊息內，並會佔用你可用的 token 數量。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 我們想要調用的函數名稱。 

`description` - 函數工作方式的描述。在這裡具體且清晰是很重要的 

`parameters` - 你希望模型在回應中產生的值和格式列表 


`type` - 屬性將儲存的資料類型 

`properties` - 模型將用於回應的具體值列表 


`name` - 模型將在其格式化回應中使用的屬性名稱 

`type` - 該屬性的資料類型 

`description` - 該具體屬性的描述 


<strong>可選</strong>

`required` - 函數調用必須的必要屬性 


### 呼叫函數
定義函數後，我們現在需要在呼叫聊天補全 API 時包含它。我們通過在請求中添加 `functions` 來實現這一點。在這個例子中是 `functions=functions`。

也有一個選項可以將 `function_call` 設定為 `auto`。這意味著我們將讓大型語言模型根據用戶訊息決定應該呼叫哪個函數，而不是我們自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


而家我哋睇下回應，睇下佢點樣格式化： 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以見到調用咗個函數名，並且從用戶訊息入面，LLM能夠搵到啱嘅數據去匹配函數嘅參數。


## 3.將函數調用整合到應用程式中。


在我們測試了來自 LLM 的格式化回應後，現在我們可以將其整合到應用程式中。

### 管理流程

要將其整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 Open AI 服務，並將訊息存儲在名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函數，用來呼叫 Microsoft Learn API 以取得課程清單： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們將先查看模型是否想調用某個函數。之後，我們將創建一個可用的函數並將其與被調用的函數相匹配。
然後，我們將擷取函數的參數並將其對應到來自大型語言模型的參數。

最後，我們會附加函數調用訊息和 `search_courses` 訊息返回的值。這樣就能提供大型語言模型所有需要的資訊，
用以自然語言回應用戶。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將把更新後的訊息發送給 LLM，這樣我們就能收到自然語言的回應，而不是 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代碼挑戰 

很棒的工作！要繼續學習 Azure Open AI 函數調用，你可以建立：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函數參數，可能幫助學習者找到更多課程。你可以在此找到可用的 API 參數： 
 - 建立另一個函數調用，從學習者那獲取更多資訊，如他們的母語 
 - 當函數調用和/或 API 調用未返回任何合適課程時，建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
